In [70]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf


In [71]:
variable_names = [
    'sheet', 'chain', 'co_owned', 'state',
    'southj', 'centralj', 'northj', 'pa1', 'pa2', 'shore',
    'ncalls', 'empft', 'emppt', 'nmgrs', 'wage_st',
    'inctime', 'firstinc', 'bonus', 'pctaff', 'meals',
    'open', 'hrsopen', 'psoda', 'pfry', 'pentree', 'nregs', 'nregs11',
    'type2', 'status2', 'date2', 'ncalls2',
    'empft2', 'emppt2', 'nmgrs2', 'wage_st2',
    'inctime2', 'firstinc2', 'special2', 'meals2',
    'open2r', 'hrsopen2', 'psoda2', 'pfry2', 'pentree2', 'nregs2', 'nregs112',
]

DATA_URL = "https://raw.githubusercontent.com/ZuiTu00/Replication-card-Krueger-1994/main/data/raw/public.dat"
LOCAL_FILE = "public.dat"

import os
if os.path.exists(LOCAL_FILE):
    filepath = LOCAL_FILE
else:
    filepath = DATA_URL

df = pd.read_csv(
    filepath,
    sep=r'\s+',
    header=None,
    names=variable_names,
    na_values='.',
    engine='python'
)

# Construct FTE: Full-Time Equivalent = full-time + managers + 0.5 * part-time
df['fte'] = df['empft'] + df['nmgrs'] + 0.5 * df['emppt']
df['fte2'] = df['empft2'] + df['nmgrs2'] + 0.5 * df['emppt2']
df['state_label'] = df['state'].map({1: 'NJ', 0: 'PA'})

print(f"Data loaded: {df.shape[0]} stores, {df.shape[1]} variables")
df.head()

Data loaded: 410 stores, 49 variables


,sheet,chain,co_owned,state,southj,centralj,northj,pa1,pa2,shore,...,open2r,hrsopen2,psoda2,pfry2,pentree2,nregs2,nregs112,fte,fte2,state_label
0,46,1,0,0,0,0,0,1,0,0,...,6.5,16.5,1.03,NaN,0.94,4.0,4.0,40.50,24.0,PA
1,49,2,0,0,0,0,0,1,0,0,...,10.0,13.0,1.01,0.89,2.35,4.0,4.0,13.75,11.5,PA
2,506,2,1,0,0,0,0,1,0,0,...,11.0,11.0,0.95,0.74,2.33,4.0,3.0,8.50,10.5,PA
3,56,4,1,0,0,0,0,1,0,0,...,10.0,12.0,0.92,0.79,0.87,2.0,2.0,34.00,20.0,PA
4,61,4,1,0,0,0,0,1,0,0,...,10.0,12.0,1.01,0.84,0.95,2.0,2.0,24.00,35.5,PA


In [72]:
def calc_stats(df, var, state):
    x = df.loc[df['state'] == state, var].dropna()
    return x.mean(), x.std() / np.sqrt(len(x)), len(x)

def t_stat(df, var):
    nj = df.loc[df['state'] == 1, var].dropna()
    pa = df.loc[df['state'] == 0, var].dropna()
    nj_se = nj.std() / np.sqrt(len(nj))
    pa_se = pa.std() / np.sqrt(len(pa))
    diff = nj.mean() - pa.mean()
    se = np.sqrt(nj_se**2 + pa_se**2)
    return diff / se if se > 0 else np.nan

# === 1. Distribution of Store Types (percentages) ===
print("=" * 65)
print("TABLE 2 REPLICATION: Means of Key Variables")
print("=" * 65)
print(f"{'Variable':<35} {'NJ':>10} {'PA':>10} {'t':>8}")
print("-" * 65)
print("\n1. Distribution of Store Types (percentages):")

chain_map = {1: 'Burger King', 2: 'KFC', 3: 'Roy Rogers', 4: "Wendy's"}
for code, name in chain_map.items():
    df['_tmp'] = (df['chain'] == code).astype(int) * 100
    nj_m, _, _ = calc_stats(df, '_tmp', 1)
    pa_m, _, _ = calc_stats(df, '_tmp', 0)
    t = t_stat(df, '_tmp')
    print(f"   {name:<32} {nj_m:>8.1f}   {pa_m:>8.1f}   {t:>6.1f}")

df['_tmp'] = df['co_owned'] * 100
nj_m, _, _ = calc_stats(df, '_tmp', 1)
pa_m, _, _ = calc_stats(df, '_tmp', 0)
t = t_stat(df, '_tmp')
print(f"   {'Company-owned':<32} {nj_m:>8.1f}   {pa_m:>8.1f}   {t:>6.1f}")

# === 2. Means in Wave 1 ===
print("\n2. Means in Wave 1:")

df['pct_ft'] = df['empft'] / df['fte'] * 100
df['wage_425'] = (df['wage_st'] == 4.25).astype(int) * 100
df['full_meal'] = df['psoda'] + df['pfry'] + df['pentree']
df['bonus_pct'] = df['bonus'] * 100

wave1_vars = [
    ('fte', 'FTE employment'),
    ('pct_ft', 'Percentage full-time employees'),
    ('wage_st', 'Starting wage'),
    ('wage_425', 'Wage = $4.25 (percentage)'),
    ('full_meal', 'Price of full meal'),
    ('hrsopen', 'Hours open (weekday)'),
    ('bonus_pct', 'Recruiting bonus'),
]

for var, label in wave1_vars:
    nj_m, nj_se, _ = calc_stats(df, var, 1)
    pa_m, pa_se, _ = calc_stats(df, var, 0)
    t = t_stat(df, var)
    print(f"   {label:<32} {nj_m:>8.1f}   {pa_m:>8.1f}   {t:>6.1f}")
    print(f"   {'':<32} {'('+str(round(nj_se,2))+')':>8}   {'('+str(round(pa_se,2))+')':>8}")

# === 3. Means in Wave 2 ===
print("\n3. Means in Wave 2:")

df['pct_ft2'] = df['empft2'] / df['fte2'] * 100
df['wage_425_2'] = (df['wage_st2'] == 4.25).astype(int) * 100
df['wage_505_2'] = (df['wage_st2'] == 5.05).astype(int) * 100
df['full_meal2'] = df['psoda2'] + df['pfry2'] + df['pentree2']
df['special2_pct'] = df['special2'] * 100

wave2_vars = [
    ('fte2', 'FTE employment'),
    ('pct_ft2', 'Percentage full-time employees'),
    ('wage_st2', 'Starting wage'),
    ('wage_425_2', 'Wage = $4.25 (percentage)'),
    ('wage_505_2', 'Wage = $5.05 (percentage)'),
    ('full_meal2', 'Price of full meal'),
    ('hrsopen2', 'Hours open (weekday)'),
    ('special2_pct', 'Recruiting bonus'),
]

for var, label in wave2_vars:
    nj_m, nj_se, _ = calc_stats(df, var, 1)
    pa_m, pa_se, _ = calc_stats(df, var, 0)
    t = t_stat(df, var)
    print(f"   {label:<32} {nj_m:>8.1f}   {pa_m:>8.1f}   {t:>6.1f}")
    print(f"   {'':<32} {'('+str(round(nj_se,2))+')':>8}   {'('+str(round(pa_se,2))+')':>8}")

print("-" * 65)
print("Notes: Standard errors in parentheses. t = test of equality of means.")

TABLE 2 REPLICATION: Means of Key Variables
Variable                                    NJ         PA        t
-----------------------------------------------------------------

1. Distribution of Store Types (percentages):
   Burger King                          41.1       44.3     -0.5
   KFC                                  20.5       15.2      1.2
   Roy Rogers                           24.8       21.5      0.6
   Wendy's                              13.6       19.0     -1.1
   Company-owned                        34.1       35.4     -0.2

2. Means in Wave 1:
   FTE employment                       20.4       23.3     -2.0
                                      (0.51)     (1.35)
   Percentage full-time employees       32.8       35.0     -0.7
                                      (1.33)     (2.73)
   Starting wage                         4.6        4.6     -0.4
                                      (0.02)     (0.04)
   Wage = $4.25 (percentage)            30.5       32.9     -0.4
  

In [73]:
nj_before = df.loc[df['state'] == 1, 'fte'].mean()
nj_after  = df.loc[df['state'] == 1, 'fte2'].mean()
pa_before = df.loc[df['state'] == 0, 'fte'].mean()
pa_after  = df.loc[df['state'] == 0, 'fte2'].mean()

nj_change = nj_after - nj_before
pa_change = pa_after - pa_before
did_estimate = nj_change - pa_change

print("=" * 60)
print("MANUAL DID CALCULATION")
print("=" * 60)
print(f"NJ mean FTE (before): {nj_before:.2f}")
print(f"NJ mean FTE (after):  {nj_after:.2f}")
print(f"NJ change:            {nj_change:.2f}")
print(f"")
print(f"PA mean FTE (before): {pa_before:.2f}")
print(f"PA mean FTE (after):  {pa_after:.2f}")
print(f"PA change:            {pa_change:.2f}")
print(f"")
print(f"DID estimate: {did_estimate:.2f}")

MANUAL DID CALCULATION
NJ mean FTE (before): 20.44
NJ mean FTE (after):  21.03
NJ change:            0.59

PA mean FTE (before): 23.33
PA mean FTE (after):  21.17
PA change:            -2.17

DID estimate: 2.75


In [74]:
# Reshape to long format
wave1 = df[['sheet', 'state', 'fte']].copy()
wave1['post'] = 0
wave1 = wave1.rename(columns={'fte': 'employment'})

wave2 = df[['sheet', 'state', 'fte2']].copy()
wave2['post'] = 1
wave2 = wave2.rename(columns={'fte2': 'employment'})

df_long = pd.concat([wave1, wave2], ignore_index=True)
df_long = df_long.dropna(subset=['employment'])
df_long['treat_post'] = df_long['state'] * df_long['post']

# DID regression
model = smf.ols('employment ~ state + post + treat_post', data=df_long).fit()
print("=" * 60)
print("DID REGRESSION: employment ~ state + post + treat_post")
print("=" * 60)
print(model.summary2().tables[1])
print(f"\nIntercept:    {model.params['Intercept']:.2f}  (PA mean FTE before)")
print(f"state:        {model.params['state']:.2f}  (NJ-PA gap before)")
print(f"post:         {model.params['post']:.2f}  (PA change over time)")
print(f"treat_post:   {model.params['treat_post']:.2f}  (DID causal effect)")

DID REGRESSION: employment ~ state + post + treat_post
                Coef.  Std.Err.          t         P>|t|     [0.025     0.975]
Intercept   23.331169  1.071870  21.766795  1.163534e-82  21.227119  25.435219
state       -2.891761  1.193524  -2.422877  1.562199e-02  -5.234614  -0.548908
post        -2.165584  1.515853  -1.428625  1.535074e-01  -5.141160   0.809991
treat_post   2.753606  1.688409   1.630888  1.033126e-01  -0.560693   6.067905

Intercept:    23.33  (PA mean FTE before)
state:        -2.89  (NJ-PA gap before)
post:         -2.17  (PA change over time)
treat_post:   2.75  (DID causal effect)


In [75]:
model_cl = smf.ols('employment ~ state + post + treat_post', data=df_long).fit(
    cov_type='cluster',
    cov_kwds={'groups': df_long['sheet']}
)

print("=" * 60)
print("DID REGRESSION WITH CLUSTERED STANDARD ERRORS")
print("=" * 60)
print(model_cl.summary2().tables[1])

print(f"\nComparison of Standard Errors (treat_post):")
print(f"  OLS SE:       {model.bse['treat_post']:.3f}")
print(f"  Clustered SE: {model_cl.bse['treat_post']:.3f}")
print(f"  Ratio:        {model_cl.bse['treat_post'] / model.bse['treat_post']:.2f}x")

DID REGRESSION WITH CLUSTERED STANDARD ERRORS
                Coef.  Std.Err.          z         P>|z|     [0.025     0.975]
Intercept   23.331169  1.346540  17.326755  2.955386e-67  20.691999  25.970339
state       -2.891761  1.436604  -2.012914  4.412370e-02  -5.707454  -0.076068
post        -2.165584  1.218028  -1.777943  7.541330e-02  -4.552876   0.221707
treat_post   2.753606  1.306485   2.107645  3.506171e-02   0.192943   5.314269

Comparison of Standard Errors (treat_post):
  OLS SE:       1.688
  Clustered SE: 1.306
  Ratio:        0.77x


In [76]:
print("=" * 60)
print("SUMMARY OF DID ESTIMATES")
print("=" * 60)
print(f"Manual DID estimate:        {did_estimate:.2f}")
print(f"")
print(f"Regression (Treat x Post):")
print(f"  Coefficient:              {model_cl.params['treat_post']:.2f}")
print(f"  Clustered SE:             {model_cl.bse['treat_post']:.3f}")
print(f"  p-value:                  {model_cl.pvalues['treat_post']:.3f}")
print(f"")
print(f"Conclusion: The minimum wage increase in NJ did NOT reduce")
print(f"employment relative to PA. The DID estimate is positive and")
if model_cl.pvalues['treat_post'] < 0.05:
    print(f"statistically significant at the 5% level.")
else:
    print(f"not statistically significant at the 5% level.")

SUMMARY OF DID ESTIMATES
Manual DID estimate:        2.75

Regression (Treat x Post):
  Coefficient:              2.75
  Clustered SE:             1.306
  p-value:                  0.035

Conclusion: The minimum wage increase in NJ did NOT reduce
employment relative to PA. The DID estimate is positive and
statistically significant at the 5% level.
